# Step 3: Baseline 1D CNN

Train a lightweight 3-layer CNN with:
- SGD optimizer → establishes lower-bound performance
- Adam optimizer → first comparison data point

Expected: SGD 88-91% accuracy / ~72% macro F1; Adam 93-95% / ~81% macro F1

In [1]:
import sys, os
from pathlib import Path
_r = Path.cwd()
while not (_r / 'src').is_dir():
    _r = _r.parent
os.chdir(_r)
if str(_r) not in sys.path:
    sys.path.insert(0, str(_r))

import torch
from src.utils.seed import set_seed
from src.utils.device import get_device
from src.models.baseline_cnn import BaselineCNN
from src.training.focal_loss import FocalLoss, compute_class_frequencies
from src.training.trainer import Trainer
from src.training.callbacks import EarlyStopping, ModelCheckpoint
from src.optimizers.optimizer_factory import build_optimizer

In [2]:
set_seed(42)
device = get_device()

Using GPU: NVIDIA GeForce RTX 4070 Laptop GPU


In [3]:
from src.data.loader import load_mitbih
from src.data.preprocessor import preprocess
from src.data.dataset import build_dataloaders

X_train_raw, y_train_raw, X_test_raw, y_test_raw = load_mitbih()
splits = preprocess(X_train_raw, y_train_raw, X_test_raw, y_test_raw, val_fraction=0.15)
train_loader, val_loader, test_loader = build_dataloaders(splits, batch_size=128)
print(f'Data ready — train: {splits["X_train"].shape}, val: {splits["X_val"].shape}, test: {splits["X_test"].shape}')

Data ready — train: (307870, 187, 1), val: (13134, 187, 1), test: (21892, 187, 1)


In [4]:
# Train Baseline CNN + SGD
model_sgd = BaselineCNN().to(device)
optimizer_sgd = build_optimizer(model_sgd, 'sgd')
# Use pre-SMOTE y_train_raw: post-SMOTE frequencies are all ~0.2 → uniform alpha, defeating class weighting
focal_loss = FocalLoss(gamma=2.0, class_frequencies=compute_class_frequencies(y_train_raw))
trainer_sgd = Trainer(model_sgd, optimizer_sgd, focal_loss, device,
                      callbacks=[EarlyStopping(15), ModelCheckpoint(model_name='baseline_cnn', optimizer_name='sgd')])
history_sgd = trainer_sgd.fit(train_loader, val_loader)

Epoch   1/100 | loss 0.2433/0.1188 | f1 0.4557/0.3192 | 5.1s


Epoch   2/100 | loss 0.1049/0.0840 | f1 0.6310/0.3350 | 4.9s


Epoch   3/100 | loss 0.0734/0.0699 | f1 0.6584/0.3421 | 5.3s


Epoch   4/100 | loss 0.0580/0.0567 | f1 0.6733/0.3185 | 5.5s


Epoch   5/100 | loss 0.0488/0.0559 | f1 0.6820/0.3182 | 5.7s


...

Epoch  25/100 | loss 0.0158/0.0305 | f1 0.7262/0.3805 | 6.1s


Epoch  26/100 | loss 0.0154/0.0305 | f1 0.7292/0.3876 | 5.2s


Epoch  27/100 | loss 0.0149/0.0290 | f1 0.7325/0.3822 | 4.9s


Epoch  28/100 | loss 0.0146/0.0309 | f1 0.7356/0.3655 | 4.4s


Epoch  29/100 | loss 0.0143/0.0292 | f1 0.7394/0.3983 | 5.1s
Early stopping at epoch 29.


In [5]:
# Train Baseline CNN + Adam
set_seed(42)
model_adam = BaselineCNN().to(device)
optimizer_adam = build_optimizer(model_adam, 'adam')
trainer_adam = Trainer(model_adam, optimizer_adam, focal_loss, device,
                       callbacks=[EarlyStopping(15), ModelCheckpoint(model_name='baseline_cnn', optimizer_name='adam')])
history_adam = trainer_adam.fit(train_loader, val_loader)

Epoch   1/100 | loss 0.0620/0.0345 | f1 0.6678/0.4293 | 4.6s


Epoch   2/100 | loss 0.0231/0.0344 | f1 0.7370/0.3909 | 4.6s


Epoch   3/100 | loss 0.0177/0.0213 | f1 0.7938/0.5295 | 4.7s


Epoch   4/100 | loss 0.0150/0.0447 | f1 0.8163/0.3828 | 4.9s


Epoch   5/100 | loss 0.0135/0.0249 | f1 0.8397/0.4574 | 5.3s


...

Epoch  59/100 | loss 0.0036/0.0185 | f1 0.9452/0.5901 | 5.4s


Epoch  60/100 | loss 0.0036/0.0174 | f1 0.9451/0.6299 | 5.2s


Epoch  61/100 | loss 0.0035/0.0186 | f1 0.9466/0.6527 | 5.3s


Epoch  62/100 | loss 0.0032/0.0183 | f1 0.9492/0.7207 | 5.2s


Epoch  63/100 | loss 0.0032/0.0138 | f1 0.9486/0.7049 | 5.2s
Early stopping at epoch 63.
